# Phase 3: Feature Engineering & Customer Intelligence (RFM Analysis)
## Retail Demand & Customer Intelligence Platform

---

### Project Context & Business Rationale
In enterprise e-commerce platforms like Olist, raw relational databases are designed for operational transaction processing (OLTP)—optimized for row-level inserts, transactional ACID guarantees, and low write contention. However, raw relational tables cannot be directly ingested by machine learning algorithms or executive business intelligence dashboards.

To extract predictive and behavioral signals, we must execute **Feature Engineering**: transforming normalized transactional logs into structured, customer-level feature representations.

### Notebook Objectives:
1. **Connect to Supabase PostgreSQL**: Ingest 5 core normalized transactional entities (`orders`, `customers`, `order_items`, `payments`, and `reviews`).
2. **Build the Base Analytical Frame**: Construct a denormalized transaction matrix focused strictly on completed (`delivered`) customer orders.
3. **Engineer the RFM Foundation**:
   - **Recency (R)**: Days elapsed since the customer's last purchase relative to a fixed snapshot date.
   - **Frequency (F)**: Total number of distinct completed orders per real consumer.
   - **Monetary (M)**: Total gross monetary spend per customer.
4. **Engineer Multi-Dimensional Domain Features**:
   - **Average Order Value (AOV)**: Unit economic spend per transaction.
   - **Customer Sentiment**: Average historical review rating mapped across order feedback.
   - **Fulfillment Friction**: Average delivery delay relative to promised estimated delivery dates.
   - **Customer Geography**: Resolved geographic location reflecting the customer's most recent order.
5. **Quantile Discretization & Scoring**: Transform continuous metrics into standardized 1–4 quartile scores (`R_score`, `F_score`, `M_score`, and composite `RFM_Score`).
6. **Rule-Based Customer Segmentation**: Establish an interpretable, baseline behavioral segmentation (Champions, Loyal Customers, At Risk, Lost, New Customers, Potential Loyalists).
7. **Feature Store Export**: Serialize the cleaned, engineered customer feature matrix to `../data/customer_features.csv` for downstream unsupervised clustering and predictive modeling.

---

## Section 0: Setup and Supabase Database Connection

### Why We Connect to the Central Database
In production machine learning pipelines, feature extraction workflows ingest data directly from the central data warehouse or operational replica (in our platform, Supabase PostgreSQL). This guarantees:
- **Data Freshness**: Capturing the most up-to-date transaction records rather than relying on stale local exports.
- **Auditability & Traceability**: Connecting directly to the system of record with consistent credentials managed via environment variables (`.env`).
- **Scalability**: Enabling selective filtering and SQL push-down operations where appropriate.

We use SQLAlchemy alongside `pandas.read_sql` to stream tables into memory, while configuring visualization libraries (`matplotlib`, `seaborn`) to standard reporting themes.

In [ ]:
# -----------------------------------------------------------------------------
# Section 0: Setup and Supabase Database Connection
# -----------------------------------------------------------------------------
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, text
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configure visualization styling for clear, publication-quality figures
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('crest')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

# Load environment variables from root .env
load_dotenv()
engine = create_engine(os.getenv('SUPABASE_URL'))

# Ingest the 5 primary transactional tables from Supabase PostgreSQL
print("Fetching core transactional tables from Supabase...")
orders = pd.read_sql("SELECT * FROM orders", engine)
customers = pd.read_sql("SELECT * FROM customers", engine)
order_items = pd.read_sql("SELECT * FROM order_items", engine)
payments = pd.read_sql("SELECT * FROM payments", engine)
reviews = pd.read_sql("SELECT * FROM reviews", engine)

print("Data ingestion completed successfully:")
print(f" - Orders:      {orders.shape[0]:,} rows, {orders.shape[1]} columns")
print(f" - Customers:   {customers.shape[0]:,} rows, {customers.shape[1]} columns")
print(f" - Order Items: {order_items.shape[0]:,} rows, {order_items.shape[1]} columns")
print(f" - Payments:    {payments.shape[0]:,} rows, {payments.shape[1]} columns")
print(f" - Reviews:     {reviews.shape[0]:,} rows, {reviews.shape[1]} columns")

## Section 1: What is Feature Engineering?

### Transforming Raw Data into Machine Learning Signal
**Feature Engineering** is the practice of extracting, transforming, and combining raw relational data into meaningful numeric, categorical, and temporal attributes that expose predictive signals to machine learning algorithms.

Raw transactional records cannot be fed directly into models like K-Means, XGBoost, or Random Forests:
- An order timestamp string like `"2018-05-15 14:22:08"` carries high cardinality and zero direct correlation with customer value.
- An arbitrary transaction UUID like `"e481f51cbdc54678b7cc49136f2d6af7"` is simply a database key.
- Dispersed payment rows and item-level details cannot be ingested by models expecting one unified vector per customer.

Through feature engineering, we extract behavioral signals from operational noise:
- **Raw Timestamps** $\longrightarrow$ **Recency** (Elapsed days since last interaction)
- **Transaction History** $\longrightarrow$ **Frequency** (Order cadence and repurchase velocity)
- **Payment Logs** $\longrightarrow$ **Monetary Value** (Gross financial contribution)

### The RFM Framework: A 1970s Direct Marketing Powerhouse in Modern Tech
The **RFM (Recency, Frequency, Monetary)** model originated in the 1970s for catalog and direct mail marketing (Bult & Wansbeek, 1995; Arthur Hughes, 1994). Before machine learning, direct mail firms needed an empirical heuristic to determine which households should receive expensive physical catalogs to maximize return on investment (ROI).

Fifty years later, RFM remains a cornerstone of customer analytics at global tech leaders like **Amazon, Alibaba, and Shopify** because of the **Pareto Principle (80/20 Rule)**: in retail marketplaces, roughly 80% of revenue is generated by the top 20% of loyal, repeat customers.

The three pillars capture customer engagement across time, action, and capital:
1. **Recency (R)**: How recently did the customer last buy? (Measures churn risk and re-engagement propensity)
2. **Frequency (F)**: How many times has the customer bought? (Measures brand affinity and habituation)
3. **Monetary (M)**: How much has the customer spent in total? (Measures purchasing power and customer lifetime value)

In [ ]:
# -----------------------------------------------------------------------------
# Section 1: Verifying Raw Entity Structure
# -----------------------------------------------------------------------------
# Inspecting raw order timestamps and payment records before transformation
print("Sample raw orders data (transaction grain):")
display(orders[['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp']].head(3))

print("\nSample raw payments data (transaction tender grain):")
display(payments[['order_id', 'payment_sequential', 'payment_type', 'payment_value']].head(3))

## Section 2: Building the Base Analytical Frame

### Relational Merge Strategy: Why We Build One Wide DataFrame
Relational database schemas follow Third Normal Form (3NF) to eliminate data redundancy and prevent write anomalies during online transaction processing. However, analytical and machine learning workflows require a denormalized, wide table structured at the entity level (here, one row per transaction or customer).

Our merge strategy follows four strict principles:
1. **Delivered Orders Only**: We filter strictly for `order_status == 'delivered'`. In e-commerce analytics, orders with status `canceled`, `unavailable`, `invoiced`, or `shipped` have not completed the fulfillment lifecycle. Including unfulfilled orders would falsely inflate monetary spend and corrupt delivery cycle calculations.
2. **Chronological Parsing**: We cast `order_purchase_timestamp` to datetime to enable temporal calculations.
3. **Multi-Way Relational Merge**:
   $$\text{orders (delivered)} \xrightarrow{\text{customer\_id}} \text{customers} \xrightarrow{\text{order\_id}} \text{order\_items} \xrightarrow{\text{order\_id}} \text{payments}$$
4. **Feature Selection**: We isolate key transactional features: `customer_unique_id`, `order_id`, `order_purchase_timestamp`, `payment_value`, `price`, `freight_value`.

### CRITICAL INTERVIEW DISTINCTION: `customer_id` vs `customer_unique_id` in Olist
In the Olist schema, there are two distinct customer identifiers that are frequently tested in data science interviews:
- **`customer_id`**: An ephemeral, transaction-level key generated afresh for each order checkout session.
- **`customer_unique_id`**: The true, persistent physical person identifier tied to an individual's account registry.

> **Interview Alert**: If a data scientist mistakenly groups by `customer_id`, they would calculate a 0% repeat buyer rate and conclude that 100% of customers churn after a single order! To analyze customer lifetime behavior, retention, and repeat purchases, you MUST group by `customer_unique_id`.

In [ ]:
# -----------------------------------------------------------------------------
# Section 2: Building the Base Analytical Frame (df_base)
# -----------------------------------------------------------------------------
# 1. Filter for delivered orders only
delivered_orders = orders[orders['order_status'] == 'delivered'].copy()
print(f"Total delivered orders: {len(delivered_orders):,} (out of {len(orders):,} total orders)")

# 2. Convert order_purchase_timestamp to datetime
delivered_orders['order_purchase_timestamp'] = pd.to_datetime(delivered_orders['order_purchase_timestamp'])

# 3. Merge: delivered_orders -> customers -> order_items -> payments
df_base = (
    delivered_orders
    .merge(customers, on='customer_id', how='inner')
    .merge(order_items, on='order_id', how='inner')
    .merge(payments, on='order_id', how='inner')
)

# 4. Keep analytical columns
base_columns = [
    'customer_unique_id',
    'order_id',
    'order_purchase_timestamp',
    'payment_value',
    'price',
    'freight_value'
]
df_base = df_base[base_columns].copy()

print(f"Base Analytical Frame (df_base) successfully assembled:")
print(f" - Total Records (Item-Payment Grain): {df_base.shape[0]:,}")
print(f" - Unique Customers (customer_unique_id): {df_base['customer_unique_id'].nunique():,}")
print(f" - Unique Orders (order_id): {df_base['order_id'].nunique():,}")
df_base.head()

## Section 3: Recency (R)

### Defining and Measuring Recency
**Recency** measures how recently a customer made a purchase on the platform relative to an analytical reference point. It is widely recognized as the single strongest behavioral predictor of future customer re-engagement:
- A customer who bought yesterday is far more likely to open a promotional email, return to the platform, and re-purchase than a customer who last transacted 18 months ago.
- As recency days increase, customer interest and brand recall decay exponentially.

### The Concept of the "Snapshot Date"
In an active production environment, recency is calculated relative to `datetime.now()`. However, because Olist is a historical static dataset spanning 2016 to 2018, measuring recency against today's date would make every customer appear years dormant, collapsing all recency variance.

To accurately simulate a production deployment, we define a **Snapshot Date** (observation cutoff):
$$\text{Snapshot Date} = \max(\text{order\_purchase\_timestamp}) + 1\text{ day}$$

Adding 1 day ensures that the most recent customer has a non-zero recency ($R \ge 1$ day) and eliminates negative timedelta values.

$$\text{Recency (Days)} = (\text{Snapshot Date} - \text{Last Purchase Date})_{\text{days}}$$

In [ ]:
# -----------------------------------------------------------------------------
# Section 3: Recency Calculation and Visualization
# -----------------------------------------------------------------------------
# 1. Establish the snapshot date (1 day after the latest transaction)
snapshot_date = df_base['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print(f"Latest Recorded Purchase: {df_base['order_purchase_timestamp'].max()}")
print(f"Analytical Snapshot Date:  {snapshot_date}")

# 2. Find the last purchase date for each unique customer
last_purchase = df_base.groupby('customer_unique_id')['order_purchase_timestamp'].max().reset_index()
last_purchase.columns = ['customer_unique_id', 'last_purchase_date']

# 3. Compute recency in elapsed days
last_purchase['recency_days'] = (snapshot_date - last_purchase['last_purchase_date']).dt.days

# Summary statistics
print("\nRecency Summary Statistics (Days):")
print(last_purchase['recency_days'].describe())

# 4. Plot histogram of recency_days
plt.figure(figsize=(10, 5))
sns.histplot(last_purchase['recency_days'], bins=50, kde=True, color='#2b5c8f')
plt.axvline(last_purchase['recency_days'].median(), color='red', linestyle='--', 
            label=f"Median: {last_purchase['recency_days'].median():.0f} days")
plt.axvline(last_purchase['recency_days'].mean(), color='orange', linestyle='-', 
            label=f"Mean: {last_purchase['recency_days'].mean():.0f} days")

plt.title("Distribution of Customer Recency (Days Since Last Purchase)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Recency (Days)", fontsize=12)
plt.ylabel("Number of Customers", fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

## Section 4: Frequency (F)

### Defining and Measuring Frequency
**Frequency** measures the total number of distinct purchase transactions completed by a customer across their lifetime on the platform:
$$\text{Frequency} = \text{Count of Distinct Orders per } \texttt{customer\_unique\_id}$$

Note that we count unique `order_id` values, not item rows. If a customer orders 4 items in a single checkout, that is a single shopping event ($F=1$). If they place 4 separate orders across multiple months, that demonstrates habitual repeat usage ($F=4$).

### A Known Dataset Quirk: Extremely Low Repeat Rates in Olist
An essential insight that every senior retail analyst must communicate is that **Olist has an overwhelmingly one-and-done customer base**:
- Over 96% of customers in the Olist dataset place exactly **one** order.
- Fewer than 4% of customers ever place a second order.

#### Why does this happen in real-world retail?
1. **Catalog Composition**: Olist is an aggregation marketplace for third-party merchants selling durable, infrequent goods (office furniture, power tools, car accessories, bed frames) rather than fast-moving consumer goods (groceries, toiletries, pet food).
2. **Marketplace Discovery Dynamics**: Many Brazilian shoppers found Olist sellers through search engines or marketplace syndication (Mercado Livre) to purchase a specific item without developing platform loyalty.
3. **Data Science Implication**: In standard RFM workflows, frequency is cut into quartiles. But when 96% of the data equals 1, standard percentile bins collapse into duplicate cutoffs (`ValueError`). We must employ special rank-order binning techniques in Section 7.

In [ ]:
# -----------------------------------------------------------------------------
# Section 4: Frequency Calculation and Analysis
# -----------------------------------------------------------------------------
# 1. Calculate frequency as the number of distinct orders per customer_unique_id
frequency = df_base.groupby('customer_unique_id')['order_id'].nunique().reset_index()
frequency.columns = ['customer_unique_id', 'frequency']

# 2. Calculate percentage of customers with only 1 order
single_order_count = (frequency['frequency'] == 1).sum()
total_customers = len(frequency)
single_order_pct = (single_order_count / total_customers) * 100
repeat_order_pct = 100.0 - single_order_pct

print("Customer Purchase Frequency Breakdown:")
print(f" - Single-Order Customers (F = 1): {single_order_count:,} ({single_order_pct:.2f}%)")
print(f" - Repeat Buyers          (F > 1): {total_customers - single_order_count:,} ({repeat_order_pct:.2f}%)")
print(f" - Maximum Orders by Single Buyer: {frequency['frequency'].max()}")

# 3. Frequency countplot
plt.figure(figsize=(10, 5))
freq_counts = frequency['frequency'].value_counts().sort_index()

# Group frequency >= 4 for cleaner visual representation
plot_series = pd.Series({
    '1 Order': freq_counts.get(1, 0),
    '2 Orders': freq_counts.get(2, 0),
    '3 Orders': freq_counts.get(3, 0),
    '4+ Orders': freq_counts[freq_counts.index >= 4].sum()
})

ax = sns.barplot(x=plot_series.index, y=plot_series.values, palette='Blues_r')
plt.title("Distribution of Customer Order Frequency", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Order Frequency Tier", fontsize=12)
plt.ylabel("Number of Customers (Log Scale)", fontsize=12)
plt.yscale('log')

for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(f"{int(h):,}\n({h/total_customers*100:.1f}%)",
                    (p.get_x() + p.get_width() / 2., h),
                    ha='center', va='bottom', fontsize=10, xytext=(0, 5),
                    textcoords='offset points')

plt.tight_layout()
plt.show()

## Section 5: Monetary (M)

### Defining and Measuring Monetary Value
**Monetary Value** reflects the total gross financial expenditure committed by each customer:
$$\text{Monetary Value} = \sum \text{payment\_value per } \texttt{customer\_unique\_id}$$

This metric quantifies the customer's cumulative commercial value to the platform. Customers with high monetary spend drive the bulk of operating margins, justify premium customer support tiers, and represent prime targets for high-ticket upsells.

### Distribution Skew and Visualization Strategy
Financial spend data in e-commerce exhibits extreme right-skewness (power law distribution):
- The **median spend** is typically modest (~$100–$150 BRL).
- The **mean spend** is substantially higher due to heavy long-tail corporate and luxury purchases.
- Outliers reach above $10,000+ BRL.

When plotting histograms of heavily right-skewed variables, extreme outliers cause the vast majority of normal transactions to compress into a single unreadable bin. In our visual exploration, we clip values to $2,000 BRL to inspect the core distribution shape while preserving raw, unclipped values in the analytical dataset.

In [ ]:
# -----------------------------------------------------------------------------
# Section 5: Monetary Value Calculation and Analysis
# -----------------------------------------------------------------------------
# 1. Calculate monetary value (sum of payment_value across all orders) per customer_unique_id
monetary = df_base.groupby('customer_unique_id')['payment_value'].sum().reset_index()
monetary.columns = ['customer_unique_id', 'monetary_value']

# 2. Print descriptive metrics: mean, median, 90th percentile
mean_val = monetary['monetary_value'].mean()
median_val = monetary['monetary_value'].median()
p75_val = monetary['monetary_value'].quantile(0.75)
p90_val = monetary['monetary_value'].quantile(0.90)
p99_val = monetary['monetary_value'].quantile(0.99)
max_val = monetary['monetary_value'].max()

print("Monetary Spend Distribution Metrics (BRL):")
print(f" - Mean Spend:        ${mean_val:.2f}")
print(f" - Median Spend:      ${median_val:.2f}")
print(f" - 75th Percentile:   ${p75_val:.2f}")
print(f" - 90th Percentile:   ${p90_val:.2f}")
print(f" - 99th Percentile:   ${p99_val:.2f}")
print(f" - Maximum Spend:     ${max_val:,.2f}")

# 3. Histogram of monetary_value (clipped to 2,000 for readability)
plt.figure(figsize=(10, 5))
clipped_spend = monetary['monetary_value'].clip(upper=2000)
sns.histplot(clipped_spend, bins=50, kde=True, color='#2ca02c')
plt.axvline(median_val, color='red', linestyle='--', label=f"Median: ${median_val:.2f}")
plt.axvline(mean_val, color='orange', linestyle='-', label=f"Mean: ${mean_val:.2f}")
plt.axvline(p90_val, color='purple', linestyle=':', label=f"90th Pct: ${p90_val:.2f}")

plt.title("Distribution of Customer Monetary Value (Clipped to $2,000 BRL)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Total Spend (BRL)", fontsize=12)
plt.ylabel("Number of Customers", fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

## Section 6: Merge RFM & Add Extra Features

### Enriching RFM with Operational and Behavioral Signals
While classical RFM provides a solid 3-dimensional baseline, modern retail intelligence demands broader domain features. A high-spending customer who consistently suffers multi-week delivery delays and submits 1-star reviews is at extreme risk of churn, regardless of their historical monetary score!

In this section, we construct a 360-degree customer feature matrix:
1. **Combine R, F, M**: Merge `last_purchase`, `frequency`, and `monetary` on `customer_unique_id`.
2. **Average Order Value (AOV)**:
   $$\text{AOV} = \frac{\text{monetary\_value}}{\text{frequency}}$$
   Differentiates customers who achieve high spend via frequent small orders from those making a single large purchase.
3. **Customer Review Sentiment (`avg_review_score`)**:
   - Merge reviews through `orders` and `customers` to map ratings to `customer_unique_id`.
   - Calculate each customer's historical average review score (1.0 to 5.0).
4. **Customer Geography (`customer_state`)**:
   - Customers may place orders from different addresses over time.
   - We sort orders chronologically and extract the **most recent state** for each `customer_unique_id` to guarantee one record per customer without row duplication.
5. **Fulfillment Logistics Friction (`avg_delivery_delay`)**:
   $$\text{Delivery Delay (Days)} = (\text{order\_delivered\_customer\_date} - \text{order\_estimated\_delivery\_date})_{\text{days}}$$
   - Negative values indicate early delivery (e.g., -5 days = delivered 5 days ahead of promise).
   - Positive values indicate delivery delays (e.g., +8 days = delivery was late by over a week).
   - Average per customer to measure individual fulfillment health.

In [ ]:
# -----------------------------------------------------------------------------
# Section 6: Merge RFM + Add Extra Features
# -----------------------------------------------------------------------------
# 1. Combine R, F, M into base RFM dataframe
rfm = (
    last_purchase[['customer_unique_id', 'recency_days']]
    .merge(frequency, on='customer_unique_id', how='inner')
    .merge(monetary, on='customer_unique_id', how='inner')
)

# 2. Add: avg_order_value = monetary_value / frequency
rfm['avg_order_value'] = rfm['monetary_value'] / rfm['frequency']

# 3. Merge reviews: avg_review_score per customer_unique_id from reviews + orders + customers
reviews_mapped = (
    reviews[['order_id', 'review_score']]
    .merge(orders[['order_id', 'customer_id']], on='order_id', how='inner')
    .merge(customers[['customer_id', 'customer_unique_id']], on='customer_id', how='inner')
)
cust_reviews = reviews_mapped.groupby('customer_unique_id')['review_score'].mean().reset_index()
cust_reviews.columns = ['customer_unique_id', 'avg_review_score']

rfm = rfm.merge(cust_reviews, on='customer_unique_id', how='left')
# Impute missing review scores with overall dataset mean
rfm['avg_review_score'] = rfm['avg_review_score'].fillna(rfm['avg_review_score'].mean())

# 4. Merge customer state from customers table (take the most recent state for each customer_unique_id)
cust_orders = (
    orders[['customer_id', 'order_purchase_timestamp']]
    .merge(customers[['customer_id', 'customer_unique_id', 'customer_state']], on='customer_id', how='inner')
)
# Sort chronologically by purchase timestamp and keep the latest state
cust_latest_state = (
    cust_orders.sort_values('order_purchase_timestamp')
    .drop_duplicates(subset=['customer_unique_id'], keep='last')[['customer_unique_id', 'customer_state']]
)
rfm = rfm.merge(cust_latest_state, on='customer_unique_id', how='left')

# 5. Add delivery_delay_days per order = (order_delivered_customer_date - order_estimated_delivery_date).dt.days
delivered_orders_delay = orders[orders['order_status'] == 'delivered'].copy()
delivered_orders_delay['order_delivered_customer_date'] = pd.to_datetime(delivered_orders_delay['order_delivered_customer_date'])
delivered_orders_delay['order_estimated_delivery_date'] = pd.to_datetime(delivered_orders_delay['order_estimated_delivery_date'])

delivered_orders_delay['delivery_delay_days'] = (
    delivered_orders_delay['order_delivered_customer_date'] - delivered_orders_delay['order_estimated_delivery_date']
).dt.days

cust_delay_mapped = delivered_orders_delay.merge(
    customers[['customer_id', 'customer_unique_id']], on='customer_id', how='inner'
)
avg_delay = cust_delay_mapped.groupby('customer_unique_id')['delivery_delay_days'].mean().reset_index()
avg_delay.columns = ['customer_unique_id', 'avg_delivery_delay']

rfm = rfm.merge(avg_delay, on='customer_unique_id', how='left')
# Fill missing delays (orders with null delivery dates) with 0.0
rfm['avg_delivery_delay'] = rfm['avg_delivery_delay'].fillna(0.0)

# 6. Final rfm DataFrame columns:
final_rfm_columns = [
    'customer_unique_id',
    'recency_days',
    'frequency',
    'monetary_value',
    'avg_order_value',
    'avg_review_score',
    'avg_delivery_delay',
    'customer_state'
]
rfm = rfm[final_rfm_columns].copy()

print(f"Final rfm DataFrame Shape: {rfm.shape}")
print(f"Missing Values Check:\n{rfm.isnull().sum()}")
print("\nFirst 5 Records:")
rfm.head()

## Section 7: RFM Scoring

### Quantile Binning Mechanics
Continuous metrics have wildly differing numerical ranges (e.g., recency in hundreds of days, frequency in units of 1 to 15, monetary spend in thousands of BRL). To harmonize these dimensions into an interpretable ordinal scale, we divide each metric into quartiles ($q=4$) assigned scores from 1 to 4:
- **Score 4**: Elite tier (Top 25% performers).
- **Score 1**: Laggard tier (Bottom 25% performers).

### Directional Polarity of Scores
- **Monetary (M)**: Higher spend is better $\longrightarrow$ quartile labels `[1, 2, 3, 4]`.
- **Frequency (F)**: More purchases is better $\longrightarrow$ quartile labels `[1, 2, 3, 4]`.
- **Recency (R)**: **REVERSED!** Fewer elapsed days means the purchase was more recent $\longrightarrow$ quartile labels `[4, 3, 2, 1]`. The lowest recency days receive the highest score (4).

### INTERVIEW ESSENTIAL: Why `rank(method='first')` is Required for Frequency
When calling `pd.qcut(df['col'], q=4)`, pandas attempts to establish 4 equal-sized bins based on empirical percentiles ($25\%, 50\%, 75\%$).
However, when >96% of customer records have `frequency = 1`, the 25th, 50th, and 75th percentiles are all identically equal to 1. Pandas raises `ValueError: Bin edges must be unique`.

To overcome this fundamental data science hurdle, we rank the records using `rfm['frequency'].rank(method='first')`. This assigns each tied record an incremental rank based on its first appearance, allowing `pd.qcut` to partition the dataset into 4 clean quartile buckets.

### Composite RFM Scores
- **`RFM_Score` (String Concatenation)**: Combines scores into a 3-digit archetype (e.g. `'444'` = best customers, `'111'` = dormant low spenders).
- **`RFM_Total` (Integer Sum)**: An additive index ranging from 3 to 12.

In [ ]:
# -----------------------------------------------------------------------------
# Section 7: RFM Scoring via Quantile Discretization
# -----------------------------------------------------------------------------
# 1. R_score: REVERSED (lowest recency_days = highest score 4)
rfm['R_score'] = pd.qcut(rfm['recency_days'], q=4, labels=[4, 3, 2, 1]).astype(int)

# 2. F_score: Ranked first to break ties from low-frequency distribution
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4]).astype(int)

# 3. M_score: Highest monetary_value = highest score 4
rfm['M_score'] = pd.qcut(rfm['monetary_value'], q=4, labels=[1, 2, 3, 4]).astype(int)

# 4. Composite RFM_Score (string) and RFM_Total (integer sum)
rfm['RFM_Score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)
rfm['RFM_Total'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

print("Sample RFM Scored Customers:")
display(rfm[['customer_unique_id', 'recency_days', 'frequency', 'monetary_value', 
             'R_score', 'F_score', 'M_score', 'RFM_Score', 'RFM_Total']].head(5))

# 5. Heatmap: R_score vs M_score showing Average Monetary Value
heatmap_matrix = rfm.pivot_table(
    index='R_score',
    columns='M_score',
    values='monetary_value',
    aggfunc='mean'
)

plt.figure(figsize=(8, 6))
sns.heatmap(heatmap_matrix, annot=True, fmt='.1f', cmap='Blues', cbar_kws={'label': 'Mean Monetary Spend (BRL)'})
plt.title("RFM Heatmap: Mean Monetary Value by R_score vs M_score", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Monetary Score (1=Low Spend, 4=High Spend)", fontsize=12)
plt.ylabel("Recency Score (1=Oldest, 4=Most Recent)", fontsize=12)
plt.gca().invert_yaxis() # Invert so R=4 is at the top
plt.tight_layout()
plt.show()

## Section 8: Customer Segment Labels (Rule-Based)

### Why Establish Rule-Based Segmentation Before Machine Learning?
In Phase 4, we will deploy unsupervised machine learning algorithms (K-Means, Hierarchical Clustering) to discover latent customer clusters mathematically. However, establishing an explicit **Rule-Based Baseline** in this notebook is standard practice in enterprise analytics for two key reasons:
1. **Stakeholder Alignment**: Business executives, CRM marketers, and growth teams require intuitive, deterministic definitions rather than pure distance metrics in $N$-dimensional space.
2. **Benchmark Evaluation**: Heuristic segments provide an operational baseline to evaluate whether downstream ML clusters uncover genuinely superior business distinctions.

### Segment Taxonomy & Business Playbook:
- **Champions** ($R \ge 3, F \ge 3, M \ge 3$): Best customers who purchase frequently, spend generously, and bought recently.  
  *Strategy*: VIP loyalty perks, early access to new lines, brand advocacy programs.
- **Loyal Customers** ($F \ge 3, M \ge 3$): Consistent high-value repeat spenders.  
  *Strategy*: Upsell higher-margin products, cross-sell related categories.
- **Potential Loyalists** ($R \ge 3, F \ge 2$): Recent buyers with multiple purchases showing high growth potential.  
  *Strategy*: Targeted recommendations and tier-upgrade incentives.
- **New Customers** ($R \ge 3, F = 1$): First-time buyers who placed an order recently.  
  *Strategy*: Automated welcome drip campaigns and onboarding discounts for order #2.
- **At Risk** ($R \le 2, F \ge 2$): Former repeat customers who have not ordered in a long time.  
  *Strategy*: Reactivation offers, win-back surveys, time-sensitive coupons.
- **Lost** ($R = 1$): Longest dormancy, low engagement.  
  *Strategy*: Exclude from paid retargeting to conserve ad spend; send low-cost re-engagement emails.
- **Others**: Intermediate customer profiles across other score permutations.

In [ ]:
# -----------------------------------------------------------------------------
# Section 8: Rule-Based Customer Segmentation
# -----------------------------------------------------------------------------
def segment_customer(row):
    """
    Assigns an intuitive behavioral customer segment based on R, F, and M scores.
    """
    r = row['R_score']
    f = row['F_score']
    m = row['M_score']
    
    if r >= 3 and f >= 3 and m >= 3:
        return 'Champions'
    elif f >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r <= 2 and f >= 2:
        return 'At Risk'
    elif r == 1:
        return 'Lost'
    elif r >= 3 and f == 1:
        return 'New Customers'
    elif r >= 3 and f >= 2:
        return 'Potential Loyalists'
    else:
        return 'Others'

# Apply rule-based segmentation
rfm['segment'] = rfm.apply(segment_customer, axis=1)

# Grouped stats across segments: mean recency, frequency, and monetary value
grouped_stats = rfm.groupby('segment')[['recency_days', 'frequency', 'monetary_value']].mean()
print("Grouped Statistics per Segment (Means):")
display(grouped_stats)

# Additional performance summary: customer count, total revenue, and percentages
segment_summary = rfm.groupby('segment').agg(
    customer_count=('customer_unique_id', 'count'),
    mean_recency=('recency_days', 'mean'),
    mean_frequency=('frequency', 'mean'),
    mean_monetary=('monetary_value', 'mean'),
    total_spend=('monetary_value', 'sum')
).reset_index()

segment_summary['pct_customers'] = (segment_summary['customer_count'] / len(rfm)) * 100
segment_summary['pct_spend'] = (segment_summary['total_spend'] / segment_summary['total_spend'].sum()) * 100
segment_summary = segment_summary.sort_values(by='total_spend', ascending=False)

print("\nDetailed Segment Performance Summary:")
display(segment_summary)

# Countplot of segments
plt.figure(figsize=(12, 5))
segment_order = rfm['segment'].value_counts().index
ax = sns.countplot(data=rfm, x='segment', order=segment_order, palette='viridis')
plt.title("Customer Distribution Across Behavioral Segments", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Segment", fontsize=12)
plt.ylabel("Number of Customers", fontsize=12)
plt.xticks(rotation=20)

for p in ax.patches:
    h = p.get_height()
    ax.annotate(f"{int(h):,}\n({h/len(rfm)*100:.1f}%)",
                (p.get_x() + p.get_width() / 2., h),
                ha='center', va='bottom', fontsize=10, xytext=(0, 4),
                textcoords='offset points')

plt.tight_layout()
plt.show()

## Section 9: Save to CSV

### Why We Save Feature Tables to CSV
In modern production machine learning pipelines, **Feature Engineering is decoupled from Model Training**:
1. **Eliminate Database Query Overhead**: Re-running expensive joins and group-by aggregations across raw tables on every ML training cycle introduces server load and network bottlenecks.
2. **Ensure Deterministic Reproducibility**: Serializing features to `../data/customer_features.csv` freezes the customer representation at our exact snapshot date, allowing downstream clustering (Notebook 05) and predictive models (Notebook 06) to train on consistent inputs.
3. **Decoupled Architecture**: Downstream notebooks read directly from the feature table without requiring direct database access.

In [ ]:
# -----------------------------------------------------------------------------
# Section 9: Save Feature Table to CSV
# -----------------------------------------------------------------------------
output_file = '../data/customer_features.csv'

# Ensure directory exists
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Save to CSV
rfm.to_csv(output_file, index=False)
print(f'Saved {len(rfm)} customer records to {output_file}')
print("\nCustomer Feature Matrix Column Data Types:")
print(rfm.dtypes)

## Section 10: Interview Talking Points

### 5 Key Takeaways to Discuss in Data Science Interviews

1. **The Critical Difference Between `customer_id` and `customer_unique_id` in Relational Schemas**
   - *Concept*: In Olist, `customer_id` is an order-level surrogate key generated per checkout transaction, whereas `customer_unique_id` identifies the actual physical human being across multiple purchases.
   - *Interview Talking Point*: *"When conducting customer lifetime analytics, verifying key entity granularity is paramount. If you aggregate by transaction-level foreign keys (`customer_id`), you would incorrectly conclude that 100% of customers churn after one order. Grouping strictly by `customer_unique_id` is essential to uncover true customer retention dynamics and repeat purchase rates."*

2. **Understanding the Mechanics and Limitations of the RFM Framework in Low-Repeat Marketplaces**
   - *Concept*: Recency, Frequency, and Monetary metrics capture customer recency, habituation, and financial contribution based on the Pareto Principle (80/20 rule).
   - *Interview Talking Point*: *"While textbook RFM assumes a broad distribution of repeat buyers, real-world marketplaces often exhibit sharp structural quirks. In Olist, over 96% of customers purchase only once due to the durable-goods catalog. Demonstrating how you diagnose and accommodate low-frequency distributions shows business acumen and practical experience beyond textbook theory."*

3. **Overcoming Duplicate Bin Edge Collisions in `pd.qcut` Using Rank-Order Discretization**
   - *Concept*: `pd.qcut` partitions continuous distributions into equal-frequency quantiles. When over 25% of records share identical values (e.g. `frequency = 1`), percentile cutoffs collide, raising `ValueError: Bin edges must be unique`.
   - *Interview Talking Point*: *"To discretize heavily tied data into clean quartiles without dropping records or triggering bin collisions, I apply rank-order tie-breaking (`rank(method='first')`) prior to quantile slicing. This deterministically breaks ties while preserving the underlying ordinal distribution."*

4. **Augmenting Classical RFM with Domain Intelligence (Fulfillment Friction & Review Sentiment)**
   - *Concept*: Classical RFM is purely financial and transactional; modern retail intelligence requires incorporating operational fulfillment health and sentiment signals.
   - *Interview Talking Point*: *"Classical RFM tells you what a customer spent, but not how they experienced your platform. By engineering `avg_delivery_delay` from actual vs. estimated delivery timestamps and `avg_review_score` from feedback records, we enrich RFM into a holistic 360-degree customer profile capable of detecting dissatisfied high-value customers at immediate risk of attrition."*

5. **Architectural Best Practices: Decoupling Feature Pipelines from Model Training via Feature Stores**
   - *Concept*: Persisting engineered feature matrices to serialized artifacts (like `../data/customer_features.csv`) separates heavy extraction and transformation workloads from iterative machine learning model development.
   - *Interview Talking Point*: *"In production machine learning architecture, feature engineering pipelines should be completely decoupled from model training. Persisting the engineered feature matrix to a versioned artifact guarantees experiment reproducibility, eliminates database query saturation during training, and enforces strict schema contracts across downstream modeling notebooks."*